# Cross-encoder reranking (decision 0008, stage 3)

bge-reranker-v2-m3 (BAAI, Apache-2.0, multilingual) reranks each dev query's fused top 50
(`data/rerank/dev_candidates_v1.tsv`, written by `src.retrieval.rerank prepare`). A document scores as its best passage
among its first 16 (MaxP). All code and settings are in `src/retrieval/rerank.py`; this notebook only runs it.

**Setup on Kaggle**
1. Accelerator **GPU T4 x2** (one is used), Internet **on**.
2. Add the same private dataset the dense notebook uses, holding `passages_v1.jsonl.gz`.
3. **Save Version, Save & Run All**: the run continues when the browser closes.

**Bring back:** download `rerank_results.zip` from the output and unzip it at the repository root
(`results/retrieval/runs/bge_reranker_dev_v1.trec` and its `.json`).

In [ ]:
REPO = "https://github.com/azzs11/heritagerec.git"
BRANCH = "main"
SPLIT = "dev"  # never "test" before docs/plan/analysis_plan.md is committed

In [ ]:
!git clone --depth 1 -b $BRANCH $REPO heritagerec
!pip install -q -U "sentence-transformers>=3.0"

In [ ]:
import glob
import os
import subprocess

found = glob.glob("/kaggle/input/**/passages_v1.jsonl.gz", recursive=True)
assert len(found) == 1, f"expected one passages_v1.jsonl.gz under /kaggle/input, found {found}"
PASSAGES = found[0]
os.chdir("heritagerec")
assert os.path.exists(f"data/rerank/{SPLIT}_candidates_v1.tsv"), "run `src.retrieval.rerank prepare` and merge it first"
subprocess.run(
    [
        "python",
        "-m",
        "src.retrieval.rerank",
        "score",
        "--split",
        SPLIT,
        "--queries",
        f"data/queries/{SPLIT}_v1.tsv",
        "--passages",
        PASSAGES,
        "--out",
        "/kaggle/working/results/retrieval/runs",
    ],
    check=True,
)

Check the printed passage counts; a query with 0 passages means its candidates were not found in the passage file.

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/rerank_results", "zip", "/kaggle/working", "results")
print("Download rerank_results.zip from the Output panel and unzip it at the repository root")